### Gestions des dépendances, ne pas modifier.

In [ ]:
!pip install pandas==2.1.4 matplotlib==3.7.1 seaborn==0.13.1 numpy==1.26.4

In [ ]:
!pip install tqdm

## Importation des packages et fichiers.

**On commence par importer les packages.**


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib

import pandas as pd
import numpy as np

from tqdm import tqdm

# On peut fixer des paramètres pour tout le notebook :

plt.style.use('seaborn-v0_8') # On configure le style graphique des graphs.
matplotlib.rcParams['figure.figsize'] = (15.0, 6.0) # On configure la taille des graphs.

#### Maintenant que nous avons chargé les packages et correctement initialisé le notebook, **il faut charger nos données.**

**Nous allons utiliser les données d'une entreprise de livraison de repas.**

- `df_meal` décrit le type de repas.
- `df_center` décrit le nom du centre d'approvisionnement.
- `df_food` décrit les commandes individuelles.

In [ ]:
df_meal = pd.read_csv("/content/meal_info.csv")
df_center = pd.read_csv("/content/fulfilment_center_info.csv")
df_food = pd.read_csv("/content/orders.csv")

In [ ]:
# 0n fusionne les informations en une seule dataframe :

df = pd.merge(df_food, df_center, on='center_id') # Rappelez-vous, le paramètre 'on=' est utilisé pour définir la clé de jointure.
df = pd.merge(df, df_meal, on='meal_id') # Le paramètre 'how=' par défaut, est défini comme étant 'inner'.

display(df)

## Et maintenant, la syntaxe Seaborn.

L'heure est venu de vous présenter les **visualisations habituelles avec Seaborn.** Pour plus de personnalisation, vous pouvez explorer les fonctions matplotlib, et pour des graphes interactifs et collaboratifs, Tableau et Looker Datastudio sont des logiciels dans lesquels utiliser les données préparées sur Python.

**La syntaxe principale est** `sns.graph_type(variable, paramètres)`.

* > La plupart de graphes existent aussi sur matplotlib, avec une syntaxe un peu plus complète mais la même idée : `sns.graph_type(variable, paramètres)` pour créer le graphe, avant d'en ajuster les détails.

In [ ]:
# Préparons les données, on étudie le nombre de commandes, par type de cuisine et zone d'opération.

op_table = df.groupby(['op_area','cuisine'])['num_orders'].agg('sum').to_frame().reset_index() # Nous groupons les données par zones d'opération et types de cuisine, puis nous faisons la somme des commandes, que nous convertissons en dataframe avec index réinitialisé.

display(op_table)


# On peut aussi le faire avec la méthode pivot_table :
# op_table = pd.pivot_table(df, index = ['op_area','cuisine'], values = 'num_orders', aggfunc = np.sum).reset_index()

### Les relationship plot ou graphiques de relations.

Seaborn propose une **unique fonction** `relplot()` pour afficher la relation entre **deux variables**, on peut choisir si l'on préfère un **nuage de points** (par défaut) ou un **graphe linéaire (line plot).**

Regardons la relation qu'entretiennent les variables `operation area` et `number of orders`.

**La relation est-elle linéaire ?**

#### Les scatter plots ou nuages de points.

Les nuages de points servent à visualiser **la relation entre deux variables numériques** en représentant chaque paire de valeurs par un point.

Ils sont particulièrement utiles pour détecter **des corrélations, des tendances, ou des patterns** dans les données, ainsi que pour identifier des **anomalies** ou des regroupements spécifiques de points.

> _Ce type de graphique est essentiel pour explorer la relation linéaire ou non linéaire entre les variables et pour observer la distribution des données dans un espace bidimensionnel._

In [ ]:
sns.relplot(data=op_table, x='op_area', y='num_orders') # Nous appliquons la méthode '.relplot()' pour initialiser et charger le graphique.

plt.xlabel('Operation area') # '.xlabel()' permet d'étiqueter l'axe des abscisses.
plt.ylabel('Number of orders') # La même chose pour l'axe des ordonnées.

plt.title('Est-ce que les zones d\'opération influence le nombre de ventes ?')
plt.show()

**La méthode ci-dessous, `.select_dtypes()` est importante.**

> _Elle permet de sélectionner les champs d'un DataFrame partageant le même type de données. Ainsi, en y spécifiant le paramètre `include=` suivi d'un argument (`number`, `O`) nous filtrons la donnée par type de données._

#### Super utile puisque statistiquement, **nous ne traitons pas de la même manière la donnée numérique et catégorielle.**

In [ ]:
df.select_dtypes(include='number') # Nous affichons uniquement les variables numériques de notre DataFrame.

**La méthode ci-dessous permet d'afficher les relations entre toutes les variables**. Attention, elle n'accepte que les données numériques.

In [ ]:
sns.pairplot(df[['base_price', 'checkout_price', 'op_area']]) # Visualisons les relations des variables numériques.

Comme on peut le voir, la méthode **`.pairplot()`** nous renvoie **toutes les relations des variables numériques qu'on spécifie.**

Cela permet **d'identifier visuellement** des corrélations, des tendances, ou des anomalies entre les paires de variables, tout en fournissant des distributions univariées sur la diagonale pour une analyse complète des interactions au sein du jeu de données.

In [ ]:
# Il est également possible de venir afficher des graphiques issus de valeurs stockées dans des listes.

a = list(op_table['op_area'])
b = list(op_table['num_orders'])

sns.relplot(x=a, y=b) # La variable 'zone d'opération' est en abscisse, les nombres de commandes sont en ordonnée.
plt.show()

### Un peu plus de détails sur les line charts.

**Concrètement, les line charts sont des graphiques qui relient des points de données avec des lignes droites.**


Ils permettent de visualiser **l'évolution d'une variable continue sur un axe**, généralement le temps, ou par rapport à une autre variable.

> _Ils sont particulièrement efficaces et utilisés pour identifier des tendances, des cycles ou des variations dans les données, rendant les changements progressifs ou les relations temporelles faciles à analyser._

In [ ]:
sns.relplot(data=op_table, x='op_area', y='num_orders', kind='line') # Pour configurer un line chart, il suffit de paramètrer 'kind=' par 'line'.

plt.xlabel('Operation area') # Définir l'étiquette de l'axe des abscisses.
plt.ylabel('Number of orders') # Définir l'étiquette de l'axe des ordonnées.

plt.title('Est-ce que les zones d\'opération influence le nombre de ventes ?')

plt.show()

**Ici, il faut noter que le graphe montre la moyenne des points (la tendance), avec un intervalle de confiance à 95% pour chaque valeur.**

### Scatter plots un peu plus complexes.

Comme pour d'autres types de graphes, le paramètre **`hue`** permet de **différencier les données en ajoutant une dimension supplémentaire via la couleur**, ce qui facilite la comparaison entre plusieurs sous-groupes au sein de la même visualisation.

De plus, le paramètre **`size`** permet de **varier la taille des points ou des éléments graphiques en fonction d'une variable**, rendant ainsi les catégories plus distinctes et mettant en évidence leur importance relative.


#### **Voyons cela en pratique.**

In [ ]:
sns.relplot(data=op_table, x='op_area', y='num_orders', hue="cuisine", aspect=2) # Commençons par ajouter le paramètre 'hue'.
plt.show()

In [ ]:
op_table2 = op_table.groupby(['cuisine', 'op_area'])['num_orders'].sum() # Nous créeons une table pour affiner le groupement des données.
op_table2 = op_table2.sort_values(ascending=False).to_frame() # Nous trions les valeurs.

display(op_table2)

In [ ]:
sns.relplot(data=op_table2, x='op_area', y='num_orders', hue="cuisine", aspect=2, size="cuisine") # Ajoutons le paramètre 'size'.
plt.show()

Comme on peut le voir, **ces paramètres enrichissent l'analyse visuelle en ajoutant de la profondeur et en facilitant la compréhension des relations complexes entre les variables.**